# 11강 실습: 모두의 말뭉치 JSON을 표로

국립국어원 온라인 대화 말뭉치 2021의 JSON 파일 47,421개를 Python으로 읽어, **발화 표**와 **화자 표** 두 CSV로 저장한다. 12강과 13강은 이 두 파일에서 시작한다.

- 이 노트북은 저장소 **맨 위**(`/workspaces/LDS2026`)에 두어야 `data/nikl/…` 경로가 맞는다.
- 말뭉치는 [숙제07]에서 신청해 승인받은 사람만 쓸 수 있다. 모두의 말뭉치 [말뭉치 신청 내역]에서 **온라인 대화 말뭉치 2021**을 내려받는다(`NIKL_OM_2021_v1.1_JSON.zip`, 123MB).
- 터미널에서 `mkdir -p data/nikl`로 폴더를 만들고, 탐색기의 그 폴더에 zip을 끌어다 놓은 뒤, 아래처럼 압축을 푼다. JSON 47,421개와 설명서 PDF 둘(한국어·영어)이 나온다.
- zip, 푼 파일, 오늘 만드는 CSV는 모두 `data/` 아래라 3강의 `.gitignore`에 걸려 커밋되지 않는다. 신청서에 "외부에 공개되지 않음"이라고 적었으니 **올려서도 안 된다**.
- 커밋하기 전에는 [Restart] 후 [Run All]로 위에서 아래까지 한 번에 돌려 본다.

```bash
cd /workspaces/LDS2026/data/nikl
unzip -q NIKL_OM_2021_v1.1_JSON.zip
ls NIKL_OM_2021_v1.1 | wc -l
head -n 11 NIKL_OM_2021_v1.1/MDRW2100000002.json
cd /workspaces/LDS2026
```

`head`로 본 파일에는 행도 열도 없다. 괄호와 이름이 있고, 안으로 갈수록 들여 썼다. 이것이 **JSON**이다.

파일 이름 `MDRW2100000002`는 설명서가 풀어 준다. `M` 온라인 대화, `D` 2인 대화(다자 대화는 `M`), `RW` 원시 말뭉치, `21` 2021년, 나머지 여덟 자리는 일련번호다.

## 0. 준비

네 가지를 불러온다.

- **json**: Python에 들어 있는 JSON 도구
- `glob`: 이름 패턴에 맞는 파일을 모두 찾는다. 4절에서 쓴다.
- `name`: 9강에서 쓴 `unicodedata.name`. 글자의 유니코드 이름을 알려 준다.
- **pandas**: 7강에서 쓴 표 라이브러리

In [1]:
import json
from glob import glob
from unicodedata import name
import pandas as pd

## 1. JSON을 Python의 값으로

먼저 JSON의 문법을 정리한다. 말뭉치 한 대화를 여는 발화를 JSON으로 적으면 이렇다.

```json
{"form": "하이하이", "speaker_id": "2"}
```

- **객체**(object): 중괄호 `{ }` 안에 **이름: 값** 짝을 쉼표로 늘어놓은 것. 이름은 언제나 큰따옴표로 감싼 문자열이다. 이름으로 값을 찾으니 짝의 순서에는 뜻이 없다.
- **배열**(array): 대괄호 `[ ]` 안에 값을 쉼표로 늘어놓은 것. 이름 없이 **자리**로 부르고, 첫 자리는 0이다. 순서에 뜻이 있다. 대화에서는 말한 차례다.
- 값은 여섯 가지다. 문자열(`"하이하이"`), 수(`3`), 참·거짓(`true`, `false`), 빈 값(`null`), 배열, 객체. 이 말뭉치에는 참·거짓과 빈 값이 없다.
- **`"2"`와 `2`는 다르다.** `"speaker_id": "2"`의 2는 문자열이고 `"intimacy": 3`의 3은 수다. 연도도 `"year": "2021"`, 문자열로 적혀 있다.
- 배열과 객체도 값이라서 객체 안에 배열이, 배열 안에 객체가 들어간다(중첩). 뿌리의 중괄호에서 값 하나에 닿는 길이 Python에서 값을 꺼내는 **경로**가 된다.

문법은 엄격하다. 큰따옴표만 쓰고(작은따옴표는 안 된다), 마지막 짝 뒤에 쉼표를 두지 않으며, 주석이 없다. 그 대신 어느 언어로 읽어도 같은 값이 나온다.

`s`는 Python의 문자열이고, 그 안의 글자가 JSON이다. 글자의 줄이라서 아직 이름으로 값을 꺼낼 수 없다. `json.loads`(load string)가 글을 읽어 **`dict`**로 바꾼다. `dict`는 이름(키)으로 값을 찾는 Python의 값이다.

In [2]:
s = '{"form": "하이하이", "speaker_id": "2"}'
d = json.loads(s)
print(type(s))
print(type(d))
print(d['form'])

<class 'str'>
<class 'dict'>
하이하이


JSON의 값은 Python의 값으로 이렇게 바뀐다. 이 말뭉치에 없는 값(`false`, `null`)까지 넣어 보았다.

| JSON | 객체 `{ }` | 배열 `[ ]` | 문자열 | 수 | `true`, `false` | `null` |
|---|---|---|---|---|---|---|
| Python | `dict` | `list` | `str` | `int`, `float` | `True`, `False` | `None` |

`'2021'`은 문자열, `3`은 수로 남았다. `print(v)`의 출력은 Python의 표기라서 작은따옴표와 `False`, `None`을 쓴다. **JSON이 아니다.**

In [3]:
j = '{"year": "2021", "intimacy": 3, "group": false, "note": null}'
v = json.loads(j)
print(v)

{'year': '2021', 'intimacy': 3, 'group': False, 'note': None}


조금 더 큰 JSON, 대화 하나다. 화자 둘의 배열(`speaker`)과 발화 둘의 배열(`utterance`)이 객체 안에 들었다.

작은따옴표 세 개(`'''`)로 감싸면 여러 줄짜리 문자열이 된다. 줄을 어떻게 바꾸든, 들여 쓰든 말든 **같은 JSON**이다. 따옴표 밖의 공백과 줄바꿈에는 뜻이 없다.

In [4]:
conv = '''{"id": "MDRW2100000002.1",
 "speaker": [{"id": "1", "age": "20대"}, {"id": "2", "age": "20대"}],
 "utterance": [{"form": "하이하이", "speaker_id": "2"},
               {"form": "반가워욬ㅌㅋㅋ", "speaker_id": "1"}]}'''
c = json.loads(conv)
print(c.keys())

dict_keys(['id', 'speaker', 'utterance'])


경로를 따라 한 칸씩 내려간다. 객체에서는 이름으로, 배열에서는 자리로 찾는다.

| 경로 | 얻는 값 | 찾은 것 |
|---|---|---|
| `c` | `dict` (대화) | — |
| `c['utterance']` | `list` (발화의 목록) | 이름 |
| `c['utterance'][1]` | `dict` (둘째 발화) | 자리 |
| `c['utterance'][1]['form']` | `str` | 이름 |

목록은 6강에서 파일의 줄을 돌던 그 `for`로 차례로 돈다.

문법이 틀리면 `json.loads`는 `JSONDecodeError`를 낸다. 아래 두 줄을 새 셀에서 하나씩 돌려 보자(이 노트북에는 넣지 않았다. [Run All]이 거기서 멈추기 때문이다).

```python
json.loads("{'form': '하이하이'}")
json.loads('{"form": "하이하이", "speaker_id": "2",}')
```

둘 다 `Expecting property name enclosed in double quotes`(큰따옴표로 감싼 이름이 와야 한다)를 낸다. 첫 줄은 작은따옴표가 있는 2번째 칸에서, 둘째 줄은 끝 쉼표 뒤의 `}`를 만난 36번째 칸에서 멈춘다. 까닭은 다른데 메시지는 같다. 메시지는 **어디서** 멈췄는지만 알려 준다.

In [5]:
print(type(c['utterance']))
print(type(c['utterance'][1]))
print(c['utterance'][1]['form'])
for u in c['utterance']:
    print(u['speaker_id'], u['form'])

<class 'list'>
<class 'dict'>
반가워욬ㅌㅋㅋ
2 하이하이
1 반가워욬ㅌㅋㅋ


## 2. Python의 값을 JSON으로

`json.dumps`는 거꾸로 Python의 값을 JSON 글로 적는다. 앞에서 만든 `d`를 쓴다.

- 기본값은 아스키 밖의 글자를 `\ud558`처럼 적는다. `하`의 코드 포인트 `U+D558`이다(9강). 이렇게 적어도 읽으면 같은 값이 된다.
- `ensure_ascii=False`는 글자를 그대로 적는다.
- `indent=2`는 사람이 읽기 좋게 줄을 바꾸고 두 칸씩 들여 쓴다. 말뭉치 파일이 바로 이 꼴이다.

| 함수 | 받는 것 | 돌려주는 것 |
|---|---|---|
| `json.loads(s)` | JSON 문자열 | Python의 값 |
| `json.dumps(v)` | Python의 값 | JSON 문자열 |
| `json.load(f)` | 열어 둔 파일 | Python의 값 |
| `json.dump(v, f)` | 값과 열어 둔 파일 | (파일에 적는다) |

끝에 `s`가 붙은 둘은 문자열(string)을, 붙지 않은 둘은 파일을 다룬다. 파일은 바이트로 되어 있으니 9강에서 배운 대로 `encoding='utf-8'`로 열어 글로 풀고, `json.load`가 그 글을 값으로 바꾼다.

In [6]:
print(json.dumps(d))
print(json.dumps(d, ensure_ascii=False))
print(json.dumps(d, ensure_ascii=False, indent=2))

{"form": "\ud558\uc774\ud558\uc774", "speaker_id": "2"}
{"form": "하이하이", "speaker_id": "2"}
{
  "form": "하이하이",
  "speaker_id": "2"
}


## 3. 파일 하나

국립국어원 온라인 대화 말뭉치(버전 1.1, 2023-09-27 공개)는 두 명 이상이 온라인 공간에서 주고받은 대화다. 대화 74,665건, 대화 메시지 3,069,927개, 파일 47,421개(835MB), JSON(UTF-8)이다. 개인 정보는 비식별화했다.

인용할 때는 설명서가 적어 준 꼴 그대로 쓴다. 국립국어원(2022). 국립국어원 온라인 대화 말뭉치(버전 1.1). URL: https://corpus.korean.go.kr

파일 하나를 연다. `with open(…) as f`는 6강에서 쓴 그대로다. 파일 하나가 객체 하나이고, 뿌리의 이름은 `id`, `metadata`, `document` 셋이다. `document`는 대화의 **배열**이다. 이 파일의 대화는 하나뿐이지만 584건이 든 파일도 있다.

In [7]:
path = 'data/nikl/NIKL_OM_2021_v1.1/MDRW2100000002.json'
with open(path, encoding='utf-8') as f:
    data = json.load(f)
print(data.keys())
print(data['metadata']['category'])
print(len(data['document']))

dict_keys(['id', 'metadata', 'document'])
온라인 대화 > 2인 대화
1


파일의 짜임은 이렇다. 1절의 대화는 첫 대화에서 이름 몇 개만 남긴 것이었다. 실제 파일에서는 `speaker`가 대화의 `metadata` 안에 한 겹 더 들어 있다.

```
파일 { }
├─ id
├─ metadata { }            말뭉치 이름, 분류 등 7개
└─ document [ ]
   └─ 대화 { }
      ├─ id
      ├─ metadata { }
      │  ├─ date, topic 등 5개
      │  ├─ speaker [ ]     화자마다 { }, 이름 9개
      │  └─ setting { }     관계, 친밀도, 연락 빈도
      └─ utterance [ ]      발화마다 { }, 이름 5개
```

화자는 이름 아홉 개로 적혔다. 나이는 `'20대'`처럼 **구간**을, 그것도 문자열로 적었다. `setting`은 두 사람의 관계다. 이 말뭉치에서 수로 적힌 값은 `intimacy`(친밀도) 하나뿐이다.

In [8]:
doc = data['document'][0]
print(doc['metadata']['speaker'][0])
print(doc['metadata']['setting'])

{'id': '1', 'age': '20대', 'occupation': '사무 종사자', 'sex': '여성', 'birthplace': '서울', 'principal_residence': '서울', 'current_residence': '서울', 'device': '스마트폰', 'keyboard': '2벌식(쿼티)'}
{'relation': '직장>선후배/상사-부하', 'intimacy': 3, 'contact_frequency': '주 3회 이상'}


셋째 발화를 통째로 찍는다. 발화에는 두 꼴이 있다.

- `original_form`: 원문 그대로다. 개인 정보만 `&name&`, `&tel-num&` 같은 비식별화 기호로 바꾸었다.
- `form`: 원문에서 겹친 공백, 특수 메시지(`{emoji}`, `{share:photo}` 등), 비식별화 기호를 지운 것이다.

여기서는 `&`만 지워져 `name2`가 남았다. 오늘부터 셀 것은 `form`이다.

In [9]:
print(doc['utterance'][2])

{'id': 'MDRW2100000002.1.3', 'form': 'name2님 제 이상형은', 'original_form': '&name2&님 제 이상형은', 'speaker_id': '1', 'time': '20210518 15:25'}


`pd.DataFrame`에 `dict`의 목록을 주면, `dict` 하나가 **행** 하나, 이름 하나가 **열** 하나가 된다. 발화 37개, 이름 다섯 개라서 `(37, 5)`다. 발화마다 이름이 모두 같으니 빈칸이 생기지 않는다.

표는 마지막 식으로 두지 않고 `print`로 찍었다. 슬라이드의 글자 출력과 모양을 맞추기 위해서다.

In [10]:
t = pd.DataFrame(doc['utterance'])
print(t.shape)
print(t[['speaker_id', 'form']].head(3))

(37, 5)
  speaker_id           form
0          2           하이하이
1          1        반가워욬ㅌㅋㅋ
2          1  name2님 제 이상형은


## 4. 발화 표

이제 47,421개 파일을 모두 읽는다. 먼저 파일 이름을 모은다.

- `glob`: 패턴에 맞는 파일 이름을 모두 모은다. `*`는 3강에서 쓴 그 와일드카드다. PDF 둘은 `*.json`에 맞지 않아 빠진다.
- `sorted`: `glob`이 돌려주는 순서는 정해져 있지 않다. 정렬해 두어야 누가 돌려도 같은 순서가 된다.

In [11]:
paths = sorted(glob('data/nikl/NIKL_OM_2021_v1.1/*.json'))
print(len(paths))
print(paths[0])

47421
data/nikl/NIKL_OM_2021_v1.1/MDRW2100000001.json


파일을 하나씩 열어, 그 안의 대화마다 쓸 가지 셋(`doc_id`, `speaker`, `utterance`)만 `dict` 하나에 골라 담는다. **시간이 걸리니 끝날 때까지 기다린다.** 대화 74,665건, 설명서의 수와 같다.

In [12]:
docs = []
for path in paths:
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    for doc in data['document']:
        docs.append({'doc_id': doc['id'],
                     'speaker': doc['metadata']['speaker'],
                     'utterance': doc['utterance']})
print(len(docs))

74665


나무를 표로 펼치는 두 단계다.

1. `pd.DataFrame(docs)`는 대화 하나가 한 행인 표다. `utterance` 칸에는 발화의 **목록**이 들었다. 10강의 `morphs` 열과 같은 꼴이다. `explode`가 목록을 펴서 발화를 한 행에 하나씩 놓는다.
2. 이제 칸마다 발화 하나(`dict`)가 들었다. `tolist()`로 꺼낸 `dict`의 목록을 `pd.DataFrame`에 주면 발화의 이름이 열이 된다.

`ignore_index=True`로 행 번호를 새로 매겼으니 `doc_id`가 제 행에 붙는다. 단계마다 행 하나가 대화 하나에서 발화 하나로 바뀌었다. 8강의 물음 "무엇을 하나로 셀 것인가"다.

발화 3,069,927개, 설명서의 수와 같다. 그런데 둘째와 셋째 행의 `form`이 비어 보인다.

In [13]:
rows = pd.DataFrame(docs).explode('utterance', ignore_index=True)
utterances = pd.DataFrame(rows['utterance'].tolist())
utterances['doc_id'] = rows['doc_id']
print(utterances.shape)
print(utterances[['doc_id', 'speaker_id', 'form']].head(3))

(3069927, 6)
             doc_id speaker_id   form
0  MDRW2100000001.1          1  안녕하세요
1  MDRW2100000001.1          2      ️
2  MDRW2100000001.1          1       


비어 보이는 두 칸을 9강에서 배운 대로 눈이 아니라 `len`과 `name`으로 확인한다.

둘째 칸에는 보이지 않는 글자 `U+FE0F`(VARIATION SELECTOR-16, 이체 선택자 16)가 있다. 앞 글자를 이모지로 그리라는 표시라 혼자서는 보이지 않는다. 원문의 이모지 열둘 가운데 하트 뒤에 붙은 이것만 괄호 밖에 있어서 지워지지 않았다. 셋째 칸은 정말 빈 문자열이다.

In [14]:
for form in utterances['form'].iloc[1:3]:
    print(len(form), [name(ch) for ch in form])

1 ['VARIATION SELECTOR-16']
0 []


`form`이 빈 발화는 92,087개다. 원문(`original_form`)을 보면 이모지, 사진·정보·링크 공유, 부적절 대화(`{censored}`)처럼 설명서의 특수 메시지만 있던 발화다. 남은 말이 없으니 뺀다. 2,977,840개가 남는다.

10강의 불용어처럼 **무엇을 뺐는지 적어 둔다**. "`form`이 빈 문자열인 발화 92,087개를 뺐다." 공백이나 `U+FE0F`처럼 보이지 않는 글자만 남은 발화(239개)는 빼지 않았다.

In [15]:
empty = utterances['form'] == ''
print(empty.sum())
print(utterances.loc[empty, 'original_form'].value_counts().head(2))
utterances = utterances[~empty]
print(len(utterances))

92087
original_form
{emoji}          33765
{share:photo}    33023
Name: count, dtype: int64
2977840


10강에서 약속한 대로 읽자마자 NFC로 맞춘다. NFC가 바꾸는 발화는 26개다.

예로 든 글은 13글자인데 코드 포인트가 30개였다. 9강의 NFD, 곧 음절 하나를 첫가끝 자모 두세 개로 적은 글이다. 이런 해시태그 글이 12개, 나머지 14개는 결합 부호를 얹은 이모티콘이다. NFD로 적힌 글에서는 `'순간' in form`이 `False`다. NFC로 모아야 검색과 세기가 맞는다.

In [16]:
nfc = utterances['form'].str.normalize('NFC')
diff = utterances['form'] != nfc
print(diff.sum())
before, after = utterances.loc[diff, 'form'], nfc[diff]
print(len(before.iloc[1]), len(after.iloc[1]), after.iloc[1])
utterances['form'] = nfc

26
30 13 #단하나의순간 #일몰순간


## 5. 화자 표

화자 표도 발화 표와 같은 두 단계로 만든다. 화자의 `id`는 발화 쪽 이름과 맞추어 `speaker_id`로 바꾸었다(`rename`). 화자 이름 아홉 개에 `doc_id`가 더해져 열이 10개다.

In [17]:
rows = pd.DataFrame(docs).explode('speaker', ignore_index=True)
speakers = pd.DataFrame(rows['speaker'].tolist())
speakers['doc_id'] = rows['doc_id']
speakers = speakers.rename(columns={'id': 'speaker_id'})
print(speakers.shape)
print(speakers[['doc_id', 'speaker_id', 'age', 'sex']].head(2))

(168267, 10)
             doc_id speaker_id  age sex
0  MDRW2100000001.1          1  20대  여성
1  MDRW2100000001.1          2  20대  여성


화자 번호는 대화마다 1부터 다시 센다. 그래서 '1번 화자'가 74,665명이다. 그러나 `doc_id`와 `speaker_id`의 짝은 168,267행에서 한 번도 겹치지 않는다(`duplicated`가 0). 발화의 화자를 찾으려면 **두 열을 함께** 맞춘다.

같은 사람이 여러 대화에 나왔는지는 이 말뭉치로 알 수 없다. 화자 표의 한 행은 '대화 안의 화자 하나'다.

In [18]:
print(speakers['speaker_id'].value_counts())
print(speakers.duplicated(['doc_id', 'speaker_id']).sum())

speaker_id
1    74665
2    74665
3    12174
4     5740
5      896
6      127
Name: count, dtype: int64
0


## 6. 저장하기

두 표를 CSV로 저장하고, 저장한 것을 다시 읽어 확인한다.

- 문자열 `"1"`이던 `speaker_id`가 수(`int64`)로 돌아왔다. CSV에는 값의 꼴을 가르는 약속이 없어서 `read_csv`가 짐작한다. 두 표에 같은 짐작을 하니 13강에서 이을 때는 괜찮다.
- 빈 칸(`NaN`)은 0이다. 빈 문자열은 CSV를 거치면 `NaN`으로 돌아오는데, 빈 발화를 미리 뺐다.

In [19]:
utterances.to_csv('data/nikl/utterances.csv', index=False)
speakers.to_csv('data/nikl/speakers.csv', index=False)
back = pd.read_csv('data/nikl/utterances.csv')
print(back.shape)
print(back.isna().sum().sum())
print(back['speaker_id'].dtype)

(2977840, 6)
0
int64


## 저장한 두 파일

터미널에서 `wc -l data/nikl/utterances.csv`로 세면 3,086,643줄이다. 행은 2,977,840인데 줄이 더 많은 것은 줄바꿈이 든 발화가 33,998개 있기 때문이다. `to_csv`는 그런 칸을 큰따옴표로 감싸 여러 줄에 걸쳐 적고, `read_csv`는 한 칸으로 읽는다. 이 파일을 4강의 AWK처럼 줄 단위로 다루면 발화가 잘린다.

| 파일 | 행 | 열 | 크기 |
|---|---:|---|---:|
| `data/nikl/utterances.csv` | 2,977,840 | `id`, `form`, `original_form`, `speaker_id`, `time`, `doc_id` | 약 355MB |
| `data/nikl/speakers.csv` | 168,267 | `speaker_id`, `age`, `occupation`, `sex`, … , `doc_id` | 16MB |

화자의 나이와 성별을 발화마다 붙여 두면 같은 내용을 2,977,840번 적어야 한다. 화자 표에 168,267번만 적고, 필요할 때 `doc_id`와 `speaker_id`로 찾아 붙인다(13강). JSON의 나무도 화자를 대화마다 한 번만 적었다.

두 CSV는 `data/` 아래라 GitHub에 없다. Codespace를 새로 만들었다면 zip을 다시 올리고 이 노트북을 [Run All]로 돌려 만든다.